<a href="https://colab.research.google.com/github/palemsaikumar29-afk/saipalem.github.io/blob/main/Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IK_Assignment1_The Autonomous IT Support Agent

# The Autonomous IT Support Agent

In [22]:
import os
import json
from openai import OpenAI
from google.colab import userdata
import random

In [5]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

Initialize Client
==========================================

PART 1: DEFINE THE TOOLS (BUSINESS LOGIC)
==========================================

In [6]:
# --- Already implement tool 1: Check Health ---
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [7]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

Task1:

In [8]:
# --- TASK 1: Implement the Restart Tool ---
def restart_service(server_id: str) -> str:
    """
    1. Print a message saying "-> TOOL: Restarting service..."
    2. Return a JSON string confirming the restart was successful.
       Example return: '{"status": "success", "message": "Server restarted successfully"}'
    """
    # 1. Print the tool execution message
    print(f"-> TOOL: Restarting service for {server_id}...")

    # 2. Prepare the success response dictionary
    response = {
        "status": "success",
        "message": f"Server {server_id} restarted successfully"
    }

    # 3. Return as a JSON string
    return json.dumps(response)

Task2:

In [9]:
# --- TASK 2: Implement the Escalation Tool ---
def escalate_to_engineer(summary: str) -> str:
    """
    1. Generates a unique Incident Ticket ID.
    2. Print a message saying "-> TOOL: Escalating to human..." with the Ticket ID.
    3. Return a JSON string confirming the ticket was created.
    """
    # Generate a random 5-digit incident ticket ID
    ticket_id = f"INC-{random.randint(10000, 99999)}"

    # 1. Print the tool execution message including the ticket ID
    print(f"-> TOOL: Escalating to human... Created Ticket: {ticket_id}")

    # 2. Prepare the escalation ticket response dictionary
    response = {
        "status": "escalated",
        "ticket_id": ticket_id,
        "message": f"Incident {ticket_id} successfully created and assigned to human engineer.",
        "summary": summary
    }

    # 3. Return as a JSON string
    return json.dumps(response)

In [10]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================

PART 2: DEFINE THE AGENT SCHEMA
==========================================

In [16]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 3: Define Schema for restart_service ---
    {
       "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restarts the service on a payment server when the CPU usage is critical (98%) and Memory usage hits 40%.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {
                        "type": "string",
                        "description": "The ID of the server where the service needs to be restarted,'payment-server-01'."
                    }
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 4: Define Schema for escalate_to_engineer ---
    {
       "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {
                        "type": "string",
                        "description": "A concise summary of the incident, including the server ID, health status metrics, and relevant error logs found."
                    }
                },
                "required": ["summary"]
            }
        }
    }
]

==========================================

PART 3: THE AGENT EXECUTION LOOP
==========================================

In [19]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)

                    messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": func_name,
                    "content": tool_output
                    })

                    pass # Remove this pass when done

        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break

In [23]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...]
-> TOOL: Checking health for payment-server-01...

[AI Thinking...]
-> TOOL: Restarting service for payment-server-01...

[AI Thinking...]
-> TOOL: Fetching last 50 log lines for payment-server-01...

[AI Thinking...]
-> TOOL: Escalating to human... Created Ticket: INC-91931

[AI Thinking...]

[FINAL RESPONSE]: I have successfully restarted the service on **payment-server-01** due to critical CPU usage at 98%. However, the server is still encountering issues, as indicated by recent logs, which include critical errors such as process hang and timeout waiting for a thread.

I have escalated the incident to a human engineer for further investigation. The ticket ID for this incident is **INC-91931**.


In [24]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...]
-> TOOL: Checking health for db-node-02...
-> TOOL: Fetching last 50 log lines for db-node-02...

[AI Thinking...]

[FINAL RESPONSE]: The health check for `db-node-02` shows the following:

- **CPU Usage:** 12% (Healthy)
- **Memory Usage:** 60% (Healthy)
- **Status:** Healthy

Additionally, the most recent logs indicate normal activity with no critical errors observed:

- `[INFO] Backup started`
- `[INFO] Backup completed successfully`
- `[INFO] User query executed in 12ms`
- `[INFO] Health check: OK`
- `[INFO] Replication sync active`

Since there are no issues with CPU, Memory, or critical errors in the logs, it seems that `db-node-02` is functioning correctly. If you're experiencing problems, please provide more specific details about the issues you're encountering.


In [25]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...]
-> TOOL: Checking health for auth-service-03...
-> TOOL: Fetching last 100 log lines for auth-service-03...

[AI Thinking...]
-> TOOL: Restarting service for auth-service-03...

[AI Thinking...]

[FINAL RESPONSE]: The login failures on **auth-service-03** were due to high memory usage (95%) and a critical error indicating a memory leak. The service has been successfully restarted, which should resolve the issue. 

Please monitor the service to ensure stability. If problems persist, further investigation may be required.




In [26]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...]
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 50 log lines for search-index-09...

[AI Thinking...]
-> TOOL: Escalating to human... Created Ticket: INC-96935

[AI Thinking...]

[FINAL RESPONSE]: I've escalated the issue with the search service on server **search-index-09**. The server has an error status with CPU at 10% and Memory at 15%. However, the logs indicate critical dependency errors, including connection refused messages and that the Search Engine is down.

A ticket has been created and assigned to a human engineer (Ticket ID: **INC-96935**) for immediate investigation.




In [27]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...]
-> TOOL: Checking health for frontend-node-04...
-> TOOL: Fetching last 50 log lines for frontend-node-04...

[AI Thinking...]

[FINAL RESPONSE]: The status of **frontend-node-04** is as follows:

- **CPU Usage:** 25%
- **Memory Usage:** 30%
- **Status:** Healthy

### Recent Logs:
- `[INFO] GET /home 200 OK`
- `[INFO] GET /assets/logo.png 200 OK`
- `[INFO] GET /login 200 OK`
- `[INFO] GET /api/v1/status 200 OK`
- `[INFO] Health check passed`

Everything appears to be functioning properly with no critical issues reported.
